In [0]:
"""Explicit schemas and Delta table helpers for three GitHub entities."""
from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                               BooleanType, TimestampType, DateType)

S = StringType()
L = LongType()
B = BooleanType()
T = TimestampType()

USER = StructType([
    StructField('id', L), StructField('login', S), StructField('type', S)
])
COMMIT_PERSON = StructType([
    StructField('name', S), StructField('email', S), StructField('date', S)
])
COMMIT_DETAIL = StructType([
    StructField('author', COMMIT_PERSON), StructField('committer', COMMIT_PERSON),
    StructField('message', S)
])
LABEL = StructType([StructField('id', L), StructField('name', S)])

COMMON_LINEAGE = [
    StructField('_repo', S), StructField('_ingested_at', S),
    StructField('_load_type', S), StructField('_source_endpoint', S),
    StructField('_batch_id', S), StructField('load_timestamp', S)
]

# Raw API fields are strings where GitHub reports ISO timestamps.
RAW_SCHEMAS = {
    'commits': StructType([
        StructField('sha', S), StructField('node_id', S),
        StructField('commit', COMMIT_DETAIL), StructField('author', USER),
        StructField('committer', USER), StructField('html_url', S),
        *COMMON_LINEAGE, StructField('_corrupt_record', S)
    ]),
    'issues': StructType([
        StructField('id', L), StructField('number', L), StructField('node_id', S),
        StructField('title', S), StructField('state', S),
        StructField('created_at', S), StructField('updated_at', S),
        StructField('closed_at', S), StructField('user', USER),
        StructField('pull_request', StructType([StructField('url', S)])),
        StructField('html_url', S),
        # Labels may be heterogeneous in some GitHub responses; model standard objects.
        StructField('labels', __import__('pyspark.sql.types', fromlist=['ArrayType']).ArrayType(LABEL)),
        *COMMON_LINEAGE, StructField('_corrupt_record', S)
    ]),
    'pulls': StructType([
        StructField('id', L), StructField('number', L), StructField('node_id', S),
        StructField('title', S), StructField('state', S), StructField('draft', B),
        StructField('created_at', S), StructField('updated_at', S),
        StructField('closed_at', S), StructField('merged_at', S),
        StructField('user', USER), StructField('merged', B),
        StructField('html_url', S),
        *COMMON_LINEAGE, StructField('_corrupt_record', S)
    ])
}

# Bronze carries normalized key / tracking columns and the known typed API fields.
BRONZE_SCHEMA = {
    entity: StructType([
        StructField('record_key', S, False),
        StructField('repository', S, False),
        StructField('batch_id', S, False),
        StructField('source_file', S),
        StructField('source_updated_at', T),
        StructField('raw_payload', S),
        StructField('payload_hash', S),
        StructField('load_timestamp', T),
        StructField('ingested_at', T),
    ]) for entity in RAW_SCHEMAS
}
# Bronze raw_payload retains a JSON object containing the explicitly parsed fields;
# the full original raw JSONL remains available in the volume.

SILVER_SCHEMA = {
    'commits': StructType([
        StructField('record_key', S, False), StructField('repository', S, False),
        StructField('commit_sha', S), StructField('author_user_id', L),
        StructField('commit_timestamp', T), StructField('message', S),
        StructField('batch_id', S), StructField('source_updated_at', T),
        StructField('payload_hash', S), StructField('load_timestamp', T)
    ]),
    'issues': StructType([
        StructField('record_key', S, False), StructField('repository', S, False),
        StructField('issue_id', L), StructField('issue_number', L),
        StructField('author_user_id', L), StructField('title', S),
        StructField('state', S), StructField('created_at', T),
        StructField('updated_at', T), StructField('closed_at', T),
        StructField('batch_id', S), StructField('source_updated_at', T),
        StructField('payload_hash', S), StructField('load_timestamp', T)
    ]),
    'pulls': StructType([
        StructField('record_key', S, False), StructField('repository', S, False),
        StructField('pr_id', L), StructField('pr_number', L),
        StructField('author_user_id', L), StructField('title', S),
        StructField('state', S), StructField('draft', B),
        StructField('created_at', T), StructField('updated_at', T),
        StructField('closed_at', T), StructField('merged_at', T),
        StructField('batch_id', S), StructField('source_updated_at', T),
        StructField('payload_hash', S), StructField('load_timestamp', T)
    ])
}

def ensure_entity_tables(spark, catalog='github_health'):
    """Create empty managed Delta tables using explicit schemas (never infer)."""
    from pyspark.sql import functions as F
    for layer, definitions in [('bronze', BRONZE_SCHEMA), ('silver', SILVER_SCHEMA)]:
        for entity, schema in definitions.items():
            table = f'{catalog}.{layer}.{entity}'
            if not spark.catalog.tableExists(table):
                spark.createDataFrame([], schema).write.format('delta').saveAsTable(table)
